# 6.23 - CertCF Adult Full-Region Hull Merge Diagnostics

This notebook tests the **true 104D analogue** of the 2D merge on Adult.

Instead of merging only anchor centers, it builds candidate merged sets from the **full original certified regions**:

\[
R_i = \{x : A_i x + b_i \ge 0,\ \|x-c_i\|_1 \le \varepsilon_i\}
\]

For each small local group, the merged set is the **exact convex hull of the union of the source regions** implemented in lifted LP form. This means:

- accepted merges contain their source regions **by construction**
- the merged object is full-dimensional when the source regions collectively span one
- certification is still done fresh, using LiRPA/CROWN affine lower bounds on a bounding box superset followed by an LP over the lifted merged hull


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import sys
import time

import cvxpy as cp
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import TensorDataset

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from certcf import CertCFAtlas, NearestOppositeClassClearanceStrategy
from certcf.certification.wrapping import WrappedModel
from dataset_specs import get_tabular_dataset_spec
from models.classifiers import TabularClassifier
from training.datamodules.adult import AdultDataModule
from training.lit_classifier import LitClassifier

torch.set_grad_enabled(False)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print({'device': str(DEVICE), 'seed': SEED})


In [ ]:
DATA_CFG = {
    'filepath': str(ROOT / 'data' / 'Adult' / 'raw.parquet'),
    'batch_size': 256,
    'val_fraction': 0.1,
    'test_fraction': 0.1,
    'seed': SEED,
    'num_workers': 0,
    'pca_enabled': False,
}

dm = AdultDataModule(**DATA_CFG)
dm.setup()
X_TRAIN, Y_TRAIN_TRUE = [t.numpy() for t in dm.train_ds.tensors]
X_TEST, Y_TEST_TRUE = [t.numpy() for t in dm.test_ds.tensors]

print({
    'train_shape': tuple(X_TRAIN.shape),
    'test_shape': tuple(X_TEST.shape),
    'train_class_counts': {int(c): int((Y_TRAIN_TRUE == c).sum()) for c in np.unique(Y_TRAIN_TRUE)},
    'test_class_counts': {int(c): int((Y_TEST_TRUE == c).sum()) for c in np.unique(Y_TEST_TRUE)},
})


In [ ]:
CKPT_PATH = ROOT / 'checkpoints' / 'adult_classifier' / 'best.ckpt'
SPEC = get_tabular_dataset_spec('adult')


def infer_tabular_classifier_dims_from_checkpoint(checkpoint: str | Path) -> tuple[list[int], int]:
    ckpt = torch.load(str(checkpoint), map_location='cpu', weights_only=False)
    state_dict = ckpt.get('state_dict', {})
    hidden_1 = state_dict.get('model.net.4.weight')
    output = state_dict.get('model.net.6.weight')
    if hidden_1 is None or output is None:
        raise KeyError('Could not infer hidden_dims / num_classes from the Adult checkpoint.')
    hidden_dims = [int(hidden_1.shape[1]), int(hidden_1.shape[0])]
    num_classes = int(output.shape[0])
    return hidden_dims, num_classes


def load_adult_model(checkpoint: str | Path, device: torch.device = DEVICE) -> torch.nn.Module:
    hidden_dims, num_classes = infer_tabular_classifier_dims_from_checkpoint(checkpoint)
    backbone = TabularClassifier(
        input_types=list(SPEC.input_types),
        cardinalities=list(SPEC.cardinalities),
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        dropout=0.2,
    )
    lit = LitClassifier.load_from_checkpoint(str(checkpoint), model=backbone, map_location=str(device))
    net = lit.model.eval().to(device)
    net_no_dropout = torch.nn.Sequential(*[m for m in net.net if not isinstance(m, torch.nn.Dropout)])
    return net_no_dropout.eval().to(device)


@torch.no_grad()
def predict_np(model: torch.nn.Module, x: np.ndarray, device: torch.device = DEVICE) -> tuple[np.ndarray, np.ndarray]:
    x_t = torch.from_numpy(np.asarray(x, dtype=np.float32)).to(device)
    logits = model(x_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy().astype(np.float32)
    preds = probs.argmax(axis=1)
    return preds, probs


MODEL = load_adult_model(CKPT_PATH, device=DEVICE)
Y_TRAIN_PRED, _ = predict_np(MODEL, X_TRAIN)
Y_TEST_PRED, _ = predict_np(MODEL, X_TEST)

print({
    'checkpoint': str(CKPT_PATH),
    'predicted_train_class_counts': {int(c): int((Y_TRAIN_PRED == c).sum()) for c in np.unique(Y_TRAIN_PRED)},
    'predicted_test_class_counts': {int(c): int((Y_TEST_PRED == c).sum()) for c in np.unique(Y_TEST_PRED)},
})


## Atlas and lifted-hull setup

We build one standard Adult atlas on a prediction-aligned support subset, expose the original local `L1` certified regions explicitly, and then test **true region-containing merges** via a lifted convex-hull LP.


In [ ]:
RUN_CFG = {
    'alpha': 0.45,
    'support_max_per_class': None,
    'batch_size': 256,
    'norm': 1,
    'distance_norm': 1,
    'solver_maxiter': 500,
    'query_method': 'nearest_anchor',
    'knn_query_block_size': 128,
    'knn_ref_block_size': 1024,
    'proposal_top_center_neighbors': 20,
    'proposal_neighbor_pool': 12,
    'proposal_max_triples_per_anchor': 10,
    'proposal_max_quads_per_anchor': 8,
    'top_pairs_to_certify': 220,
    'top_triples_to_certify': 120,
    'top_quads_to_certify': 80,
    'lirpa_method': 'backward',
    'lp_feasibility_tol': 1e-7,
    'sample_checks_per_region': 1,
    'accepted_merges_to_validate': 12,
}


def stratified_subsample(x: np.ndarray, y: np.ndarray, max_per_class: int | None, seed: int = SEED):
    if max_per_class is None:
        keep = np.arange(len(x))
        return x, y, keep
    rng = np.random.default_rng(seed)
    keep = []
    for cls in sorted(np.unique(y)):
        idx = np.where(y == cls)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        keep.append(np.sort(idx))
    keep = np.sort(np.concatenate(keep))
    return x[keep], y[keep], keep


X_SUPPORT, Y_SUPPORT, SUPPORT_IDX = stratified_subsample(
    X_TRAIN,
    Y_TRAIN_PRED,
    max_per_class=RUN_CFG['support_max_per_class'],
)

print({
    'support_shape': tuple(X_SUPPORT.shape),
    'support_class_counts': {int(c): int((Y_SUPPORT == c).sum()) for c in np.unique(Y_SUPPORT)},
})

DS = TensorDataset(torch.from_numpy(X_SUPPORT).float(), torch.from_numpy(Y_SUPPORT).long())
ATLAS = CertCFAtlas(
    MODEL,
    DS,
    DEVICE,
    cnn=False,
    norm=RUN_CFG['norm'],
    distance_norm=RUN_CFG['distance_norm'],
    lirpa_method='backward',
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=RUN_CFG['alpha']),
    batch_size=RUN_CFG['batch_size'],
    default_query_method=RUN_CFG['query_method'],
    solver_maxiter=RUN_CFG['solver_maxiter'],
)

t0 = time.perf_counter()
ATLAS.build(build_unions=False, verbose=True)
ATLAS_BUILD_SECONDS = time.perf_counter() - t0
print({'atlas_build_seconds': float(ATLAS_BUILD_SECONDS)})


In [ ]:
def choose_cvxpy_solver() -> str | None:
    for solver in ('CLARABEL', 'ECOS', 'SCS', 'OSQP'):
        if solver in cp.installed_solvers():
            return solver
    return None


CVXPY_SOLVER = choose_cvxpy_solver()
print({'cvxpy_solver': CVXPY_SOLVER})


def run_lirpa_box_lower_bound(model, target_label, x_L, x_U, device=DEVICE, lirpa_method='backward'):
    from auto_LiRPA import BoundedModule, BoundedTensor, PerturbationLpNorm

    x_L = np.asarray(x_L, dtype=np.float32).reshape(1, -1)
    x_U = np.asarray(x_U, dtype=np.float32).reshape(1, -1)
    x_center = 0.5 * (x_L + x_U)

    wrapped = WrappedModel(model, label=int(target_label), device=device, n_labels=2).to(device).to(torch.float32)
    wrapped.eval()

    x_center_t = torch.from_numpy(x_center).to(device)
    x_L_t = torch.from_numpy(x_L).to(device)
    x_U_t = torch.from_numpy(x_U).to(device)
    ptb = PerturbationLpNorm(norm=np.inf, x_L=x_L_t, x_U=x_U_t)
    X_bounded = BoundedTensor(x_center_t, ptb)

    bounded_model = BoundedModule(wrapped, X_bounded)
    _ = bounded_model(X_bounded)

    needed_A = defaultdict(set)
    needed_A[bounded_model.output_name[0]].add(bounded_model.input_name[0])
    _, _, A_dict = bounded_model.compute_bounds(
        x=(X_bounded,),
        method=lirpa_method,
        return_A=True,
        needed_A_dict=needed_A,
    )

    A = A_dict[bounded_model.output_name[0]][bounded_model.input_name[0]]
    lA = A['lA'].detach().cpu().numpy().reshape(-1, x_center.shape[1])
    lbias = A['lbias'].detach().cpu().numpy().reshape(-1)
    return lA[0].astype(np.float64), float(lbias[0])


def solve_problem(problem: cp.Problem, solver: str | None = CVXPY_SOLVER):
    kwargs = {'verbose': False, 'warm_start': True}
    if solver is not None:
        kwargs['solver'] = solver
    try:
        problem.solve(**kwargs)
    except Exception:
        if solver is None:
            raise
        fallback_kwargs = {'verbose': False, 'warm_start': True}
        problem.solve(**fallback_kwargs)


def bbox_stats_from_regions(regions):
    lowers = np.stack([region['center'] - region['eps'] for region in regions], axis=0)
    uppers = np.stack([region['center'] + region['eps'] for region in regions], axis=0)
    x_L = lowers.min(axis=0).astype(np.float64)
    x_U = uppers.max(axis=0).astype(np.float64)
    widths = np.maximum(x_U - x_L, 0.0)
    bbox_volume_proxy = float(np.sum(np.log1p(widths)))
    source_width_sum = float(sum(2.0 * region['eps'] * region['center'].shape[0] for region in regions))
    bbox_inflation_proxy = float(widths.sum() / max(source_width_sum, 1e-12))
    return {
        'x_L': x_L,
        'x_U': x_U,
        'bbox_width_l1': float(widths.sum()),
        'bbox_width_linf': float(widths.max()),
        'bbox_volume_proxy': bbox_volume_proxy,
        'bbox_inflation_proxy': bbox_inflation_proxy,
    }


def make_initial_region_descriptors(atlas, target_label: int):
    bd = atlas.bounds[target_label]
    regions = []
    for idx in range(len(bd['X'])):
        center = np.asarray(bd['X'][idx], dtype=np.float64)
        eps = float(bd['eps'][idx])
        regions.append({
            'region_key': f'class{target_label}_region_{idx}',
            'target_label': int(target_label),
            'source_id': int(idx),
            'source_ids': [int(idx)],
            'member_count': 1,
            'A': np.asarray(bd['lA'][idx], dtype=np.float64),
            'b': np.asarray(bd['lbias'][idx], dtype=np.float64),
            'center': center,
            'eps': eps,
        })
    return regions


def compute_topk_l1_neighbors(centers: np.ndarray, k: int, query_block_size: int, ref_block_size: int):
    if len(centers) == 0:
        return np.empty((0, 0), dtype=np.int64), np.empty((0, 0), dtype=np.float32)

    device = DEVICE if torch.cuda.is_available() else torch.device('cpu')
    centers_t = torch.from_numpy(np.asarray(centers, dtype=np.float32)).to(device)
    n = centers_t.shape[0]
    k_eff = min(int(k) + 1, n)
    neighbor_idx = np.empty((n, max(k_eff - 1, 0)), dtype=np.int64)
    neighbor_dist = np.empty((n, max(k_eff - 1, 0)), dtype=np.float32)

    for q_start in range(0, n, int(query_block_size)):
        q_end = min(n, q_start + int(query_block_size))
        q = centers_t[q_start:q_end]
        q_size = q_end - q_start

        best_vals = torch.full((q_size, k_eff), float('inf'), device=device)
        best_idx = torch.full((q_size, k_eff), -1, dtype=torch.long, device=device)

        for r_start in range(0, n, int(ref_block_size)):
            r_end = min(n, r_start + int(ref_block_size))
            r = centers_t[r_start:r_end]
            d_block = torch.cdist(q, r, p=1)

            if r_start <= q_end - 1 and r_end > q_start:
                q_idx = torch.arange(q_start, q_end, device=device)
                overlap_mask = (q_idx >= r_start) & (q_idx < r_end)
                if torch.any(overlap_mask):
                    local_rows = torch.nonzero(overlap_mask, as_tuple=False).squeeze(1)
                    local_cols = (q_idx[overlap_mask] - r_start).long()
                    d_block[local_rows, local_cols] = float('inf')

            block_idx = torch.arange(r_start, r_end, device=device).unsqueeze(0).expand(q_size, -1)
            merged_vals = torch.cat([best_vals, d_block], dim=1)
            merged_idx = torch.cat([best_idx, block_idx], dim=1)
            new_order = torch.topk(merged_vals, k=k_eff, dim=1, largest=False).indices
            best_vals = torch.gather(merged_vals, 1, new_order)
            best_idx = torch.gather(merged_idx, 1, new_order)

        neighbor_idx[q_start:q_end] = best_idx[:, 1:k_eff].detach().cpu().numpy()
        neighbor_dist[q_start:q_end] = best_vals[:, 1:k_eff].detach().cpu().numpy()

    return neighbor_idx, neighbor_dist


def build_local_region_candidates(regions, cfg):
    if len(regions) < 2:
        return pd.DataFrame(), {}

    centers = np.stack([region['center'] for region in regions], axis=0).astype(np.float32)
    neighbor_idx, neighbor_dist = compute_topk_l1_neighbors(
        centers,
        k=int(cfg['proposal_top_center_neighbors']),
        query_block_size=int(cfg['knn_query_block_size']),
        ref_block_size=int(cfg['knn_ref_block_size']),
    )
    candidate_map = {}

    def subset_max_center_dist(indices):
        subset_centers = centers[list(indices)].astype(np.float32)
        if len(indices) <= 1:
            return 0.0
        local_dmat = np.abs(subset_centers[:, None, :] - subset_centers[None, :, :]).sum(axis=2)
        return float(local_dmat.max())

    def register_subset(indices, proposal_source):
        indices = tuple(sorted({int(idx) for idx in indices}))
        if len(indices) < 2:
            return
        key = '|'.join(regions[idx]['region_key'] for idx in indices)
        if key in candidate_map:
            candidate_map[key]['proposal_sources'].add(proposal_source)
            return
        subset = [regions[idx] for idx in indices]
        bbox = bbox_stats_from_regions(subset)
        candidate_map[key] = {
            'candidate_key': key,
            'target_label': int(subset[0]['target_label']),
            'region_indices': indices,
            'regions': subset,
            'subset_size': int(len(indices)),
            'member_count_before': int(sum(region['member_count'] for region in subset)),
            'source_ids': [int(region['source_id']) for region in subset],
            'max_center_dist': subset_max_center_dist(indices),
            'proposal_sources': {proposal_source},
            **bbox,
        }

    for i in range(len(regions)):
        neighbors = [int(j) for j in neighbor_idx[i] if int(j) >= 0]
        for j in neighbors:
            register_subset((i, j), 'center_knn')

        local_pool = neighbors[:int(cfg['proposal_neighbor_pool'])]
        triple_count = 0
        for pos_a, j in enumerate(local_pool):
            for k in local_pool[pos_a + 1:]:
                register_subset((i, j, k), 'local_triple')
                triple_count += 1
                if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                    break
            if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                break

        quad_count = 0
        for pos_a, j in enumerate(local_pool):
            for pos_b, k in enumerate(local_pool[pos_a + 1:], start=pos_a + 1):
                for l in local_pool[pos_b + 1:]:
                    register_subset((i, j, k, l), 'local_quad')
                    quad_count += 1
                    if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                        break
                if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                    break
            if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                break

    candidate_df = pd.DataFrame(candidate_map.values()) if candidate_map else pd.DataFrame()
    if candidate_df.empty:
        return candidate_df, candidate_map

    candidate_df['proposal_source_count'] = candidate_df['proposal_sources'].map(len)
    candidate_df['proposal_sources'] = candidate_df['proposal_sources'].map(lambda values: ','.join(sorted(values)))
    candidate_df = candidate_df.sort_values(
        ['subset_size', 'max_center_dist', 'bbox_inflation_proxy', 'bbox_volume_proxy'],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)
    return candidate_df, candidate_map


def solve_lifted_hull_affine_min(c: np.ndarray, d_bias: float, regions, solver: str | None = CVXPY_SOLVER):
    dim = int(c.shape[0])
    K = len(regions)
    lam = cp.Variable(K, nonneg=True)
    x_parts = [cp.Variable(dim) for _ in range(K)]
    p_parts = [cp.Variable(dim, nonneg=True) for _ in range(K)]
    n_parts = [cp.Variable(dim, nonneg=True) for _ in range(K)]
    x_total = cp.Variable(dim)

    constraints = [cp.sum(lam) == 1]
    stacked = cp.vstack([cp.reshape(x_i, (1, dim), order='C') for x_i in x_parts])
    constraints.append(x_total == cp.sum(stacked, axis=0))

    for idx, region in enumerate(regions):
        center = region['center']
        A = region['A']
        b = region['b']
        eps = float(region['eps'])
        constraints.append(x_parts[idx] == lam[idx] * center + p_parts[idx] - n_parts[idx])
        constraints.append(cp.sum(p_parts[idx] + n_parts[idx]) <= lam[idx] * eps)
        constraints.append(A @ x_parts[idx] + lam[idx] * b >= 0)

    objective = cp.Minimize(c @ x_total + float(d_bias))
    problem = cp.Problem(objective, constraints)
    solve_problem(problem, solver=solver)
    status = problem.status
    value = float(problem.value) if problem.value is not None else np.nan
    return {
        'status': status,
        'feasible': status in ('optimal', 'optimal_inaccurate'),
        'opt_value': value,
        'x_opt': None if x_total.value is None else np.asarray(x_total.value, dtype=np.float64),
    }


def point_in_lifted_hull(point: np.ndarray, regions, solver: str | None = CVXPY_SOLVER, tol: float = 1e-7):
    point = np.asarray(point, dtype=np.float64)
    dim = int(point.shape[0])
    K = len(regions)
    lam = cp.Variable(K, nonneg=True)
    x_parts = [cp.Variable(dim) for _ in range(K)]
    p_parts = [cp.Variable(dim, nonneg=True) for _ in range(K)]
    n_parts = [cp.Variable(dim, nonneg=True) for _ in range(K)]
    stacked = cp.vstack([cp.reshape(x_i, (1, dim), order='C') for x_i in x_parts])

    constraints = [cp.sum(lam) == 1, cp.sum(stacked, axis=0) == point]
    for idx, region in enumerate(regions):
        center = region['center']
        A = region['A']
        b = region['b']
        eps = float(region['eps'])
        constraints.append(x_parts[idx] == lam[idx] * center + p_parts[idx] - n_parts[idx])
        constraints.append(cp.sum(p_parts[idx] + n_parts[idx]) <= lam[idx] * eps)
        constraints.append(A @ x_parts[idx] + lam[idx] * b >= -tol)

    problem = cp.Problem(cp.Minimize(0), constraints)
    solve_problem(problem, solver=solver)
    return problem.status in ('optimal', 'optimal_inaccurate')


def sample_point_from_region(region, direction: np.ndarray, solver: str | None = CVXPY_SOLVER):
    direction = np.asarray(direction, dtype=np.float64)
    dim = int(direction.shape[0])
    x = cp.Variable(dim)
    p = cp.Variable(dim, nonneg=True)
    n = cp.Variable(dim, nonneg=True)
    center = region['center']
    A = region['A']
    b = region['b']
    eps = float(region['eps'])

    constraints = [
        x == center + p - n,
        cp.sum(p + n) <= eps,
        A @ x + b >= 0,
    ]
    problem = cp.Problem(cp.Maximize(direction @ x), constraints)
    solve_problem(problem, solver=solver)
    if problem.status not in ('optimal', 'optimal_inaccurate') or x.value is None:
        return None
    return np.asarray(x.value, dtype=np.float64)


def certify_lifted_hull_candidate(model, target_label: int, candidate_obj, cfg):
    c_aff, d_aff = run_lirpa_box_lower_bound(
        model,
        target_label,
        x_L=candidate_obj['x_L'],
        x_U=candidate_obj['x_U'],
        device=DEVICE,
        lirpa_method=cfg['lirpa_method'],
    )
    lp_result = solve_lifted_hull_affine_min(c_aff, d_aff, candidate_obj['regions'])
    return {
        'certified': bool(lp_result['feasible'] and lp_result['opt_value'] > 0.0),
        'margin_lower_opt': float(lp_result['opt_value']),
        'affine_c': c_aff,
        'affine_d': float(d_aff),
        'lp_status': lp_result['status'],
        'x_opt': lp_result['x_opt'],
    }


def check_source_centers(candidate_obj, tol: float = 1e-7):
    rows = []
    for region in candidate_obj['regions']:
        margins = region['A'] @ region['center'] + region['b']
        rows.append({
            'source_id': int(region['source_id']),
            'center_margin_min': float(np.min(margins)),
            'center_feasible_direct': bool(np.min(margins) >= -tol),
            'center_feasible_in_hull': bool(point_in_lifted_hull(region['center'], candidate_obj['regions'], tol=tol)),
        })
    return pd.DataFrame(rows)


def sample_region_points(candidate_obj, cfg, seed: int = SEED):
    rng = np.random.default_rng(seed)
    rows = []
    for region in candidate_obj['regions']:
        for sample_idx in range(int(cfg['sample_checks_per_region'])):
            direction = rng.normal(size=region['center'].shape[0]).astype(np.float64)
            point = sample_point_from_region(region, direction)
            in_hull = False if point is None else bool(point_in_lifted_hull(point, candidate_obj['regions'], tol=cfg['lp_feasibility_tol']))
            rows.append({
                'source_id': int(region['source_id']),
                'sample_idx': int(sample_idx),
                'sample_found': bool(point is not None),
                'sample_in_hull': bool(in_hull),
            })
    return pd.DataFrame(rows)


def evaluate_target_class(target_label: int, atlas, cfg):
    regions = make_initial_region_descriptors(atlas, target_label)
    candidate_df, candidate_map = build_local_region_candidates(regions, cfg)

    if candidate_df.empty:
        return {
            'source_regions': regions,
            'candidate_df': candidate_df,
            'tested_df': pd.DataFrame(),
            'accepted_df': pd.DataFrame(),
            'center_checks': pd.DataFrame(),
            'sample_checks': pd.DataFrame(),
            'summary': {
                'target_label': int(target_label),
                'initial_region_count': int(len(regions)),
                'candidate_groups_tested': 0,
                'certified_candidates': 0,
                'accepted_region_containing_merges': 0,
                'final_region_count_if_replaced': int(len(regions)),
                'compression_ratio_if_replaced': 1.0,
            },
        }

    shortlisted = pd.concat([
        candidate_df.loc[candidate_df['subset_size'] == 2].head(int(cfg['top_pairs_to_certify'])),
        candidate_df.loc[candidate_df['subset_size'] == 3].head(int(cfg['top_triples_to_certify'])),
        candidate_df.loc[candidate_df['subset_size'] == 4].head(int(cfg['top_quads_to_certify'])),
    ], ignore_index=True)
    shortlisted = shortlisted.sort_values(
        ['subset_size', 'max_center_dist', 'bbox_inflation_proxy', 'bbox_volume_proxy'],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)

    tested_rows = []
    certified_candidates = []
    for _, row in shortlisted.iterrows():
        candidate_obj = candidate_map[row['candidate_key']]
        cert = certify_lifted_hull_candidate(MODEL, int(target_label), candidate_obj, cfg)
        tested_row = {
            **row.to_dict(),
            'certified': bool(cert['certified']),
            'margin_lower_opt': float(cert['margin_lower_opt']),
            'lp_status': cert['lp_status'],
        }
        tested_rows.append(tested_row)
        if cert['certified']:
            certified_candidates.append((tested_row, candidate_obj))

    tested_df = pd.DataFrame(tested_rows)
    certified_df = tested_df.loc[tested_df['certified']].copy() if not tested_df.empty else pd.DataFrame()

    certified_candidates = sorted(
        certified_candidates,
        key=lambda item: (
            -int(item[0]['subset_size']),
            float(item[0]['max_center_dist']),
            float(item[0]['bbox_inflation_proxy']),
            -float(item[0]['margin_lower_opt']),
        ),
    )

    used_source_ids = set()
    accepted_rows = []
    accepted_candidate_objs = []
    for tested_row, candidate_obj in certified_candidates:
        source_ids = [int(sid) for sid in candidate_obj['source_ids']]
        if any(sid in used_source_ids for sid in source_ids):
            continue
        accepted_rows.append(tested_row)
        accepted_candidate_objs.append(candidate_obj)
        used_source_ids.update(source_ids)

    accepted_df = pd.DataFrame(accepted_rows)
    center_checks = []
    sample_checks = []
    for merge_idx, candidate_obj in enumerate(accepted_candidate_objs[: int(cfg['accepted_merges_to_validate'])], start=1):
        cc = check_source_centers(candidate_obj, tol=cfg['lp_feasibility_tol'])
        cc['target_label'] = int(target_label)
        cc['merge_rank'] = int(merge_idx)
        cc['candidate_key'] = candidate_obj['candidate_key']
        center_checks.append(cc)

        sc = sample_region_points(candidate_obj, cfg, seed=SEED + merge_idx)
        sc['target_label'] = int(target_label)
        sc['merge_rank'] = int(merge_idx)
        sc['candidate_key'] = candidate_obj['candidate_key']
        sample_checks.append(sc)

    center_checks_df = pd.concat(center_checks, ignore_index=True) if center_checks else pd.DataFrame()
    sample_checks_df = pd.concat(sample_checks, ignore_index=True) if sample_checks else pd.DataFrame()

    accepted_region_count_reduction = int(accepted_df['subset_size'].sum() - len(accepted_df)) if not accepted_df.empty else 0
    final_region_count_if_replaced = int(len(regions) - accepted_region_count_reduction)

    summary = {
        'target_label': int(target_label),
        'initial_region_count': int(len(regions)),
        'candidate_groups_tested': int(len(tested_df)),
        'certified_candidates': int(len(certified_df)),
        'accepted_region_containing_merges': int(len(accepted_df)),
        'final_region_count_if_replaced': int(final_region_count_if_replaced),
        'compression_ratio_if_replaced': float(len(regions) / max(final_region_count_if_replaced, 1)),
    }

    return {
        'source_regions': regions,
        'candidate_df': candidate_df,
        'tested_df': tested_df,
        'accepted_df': accepted_df,
        'center_checks': center_checks_df,
        'sample_checks': sample_checks_df,
        'summary': summary,
    }


CLASS_RESULTS = {}
SOURCE_REGION_ROWS = []
CANDIDATE_GROUP_ROWS = []
TESTED_ROWS = []
ACCEPTED_ROWS = []
CENTER_CHECK_ROWS = []
SAMPLE_CHECK_ROWS = []
SUMMARY_ROWS = []

for target_label in sorted(np.unique(Y_SUPPORT)):
    result = evaluate_target_class(int(target_label), ATLAS, RUN_CFG)
    CLASS_RESULTS[int(target_label)] = result

    for region in result['source_regions']:
        SOURCE_REGION_ROWS.append({
            'target_label': int(target_label),
            'source_id': int(region['source_id']),
            'eps_i': float(region['eps']),
        })

    if not result['candidate_df'].empty:
        CANDIDATE_GROUP_ROWS.append(result['candidate_df'].copy())
    if not result['tested_df'].empty:
        TESTED_ROWS.append(result['tested_df'].copy())
    if not result['accepted_df'].empty:
        ACCEPTED_ROWS.append(result['accepted_df'].copy())
    if not result['center_checks'].empty:
        CENTER_CHECK_ROWS.append(result['center_checks'].copy())
    if not result['sample_checks'].empty:
        SAMPLE_CHECK_ROWS.append(result['sample_checks'].copy())
    SUMMARY_ROWS.append(result['summary'])

SOURCE_REGION_DF = pd.DataFrame(SOURCE_REGION_ROWS)
CANDIDATE_GROUP_DF = pd.concat(CANDIDATE_GROUP_ROWS, ignore_index=True) if CANDIDATE_GROUP_ROWS else pd.DataFrame()
TESTED_CANDIDATE_DF = pd.concat(TESTED_ROWS, ignore_index=True) if TESTED_ROWS else pd.DataFrame()
ACCEPTED_MERGE_DF = pd.concat(ACCEPTED_ROWS, ignore_index=True) if ACCEPTED_ROWS else pd.DataFrame()
CENTER_CHECK_DF = pd.concat(CENTER_CHECK_ROWS, ignore_index=True) if CENTER_CHECK_ROWS else pd.DataFrame()
SAMPLE_CHECK_DF = pd.concat(SAMPLE_CHECK_ROWS, ignore_index=True) if SAMPLE_CHECK_ROWS else pd.DataFrame()
FINAL_SUMMARY_DF = pd.DataFrame(SUMMARY_ROWS).sort_values('target_label').reset_index(drop=True)

SOURCE_REGION_SUMMARY_DF = (
    SOURCE_REGION_DF.groupby('target_label')['eps_i']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .rename(columns={'count': 'n_source_regions'})
    .reset_index()
)

SUCCESS_BY_SIZE_DF = (
    TESTED_CANDIDATE_DF.groupby(['target_label', 'subset_size'])['certified']
    .agg(tested='count', certified='sum')
    .reset_index()
    if not TESTED_CANDIDATE_DF.empty else pd.DataFrame()
)
if not SUCCESS_BY_SIZE_DF.empty:
    SUCCESS_BY_SIZE_DF['certification_success_rate'] = SUCCESS_BY_SIZE_DF['certified'] / SUCCESS_BY_SIZE_DF['tested']

ACCEPTED_DISPLAY_DF = ACCEPTED_MERGE_DF[[
    'target_label', 'subset_size', 'member_count_before', 'source_ids', 'certified',
    'margin_lower_opt', 'bbox_volume_proxy', 'max_center_dist'
]].copy() if not ACCEPTED_MERGE_DF.empty else pd.DataFrame()

display(SOURCE_REGION_SUMMARY_DF)
if not TESTED_CANDIDATE_DF.empty:
    display(TESTED_CANDIDATE_DF.head(30))
if not ACCEPTED_DISPLAY_DF.empty:
    display(ACCEPTED_DISPLAY_DF.head(30))
display(FINAL_SUMMARY_DF)
if not SUCCESS_BY_SIZE_DF.empty:
    display(SUCCESS_BY_SIZE_DF)
if not CENTER_CHECK_DF.empty:
    display(CENTER_CHECK_DF.head(30))
if not SAMPLE_CHECK_DF.empty:
    display(SAMPLE_CHECK_DF.head(30))


In [ ]:
GLOBAL_SUMMARY_DF = pd.DataFrame([{
    'atlas_build_seconds': float(ATLAS_BUILD_SECONDS),
    'total_initial_regions': int(FINAL_SUMMARY_DF['initial_region_count'].sum()),
    'total_candidate_groups_tested': int(FINAL_SUMMARY_DF['candidate_groups_tested'].sum()),
    'total_certified_candidates': int(FINAL_SUMMARY_DF['certified_candidates'].sum()),
    'total_accepted_region_containing_merges': int(FINAL_SUMMARY_DF['accepted_region_containing_merges'].sum()),
    'total_final_regions_if_replaced': int(FINAL_SUMMARY_DF['final_region_count_if_replaced'].sum()),
    'global_compression_ratio_if_replaced': float(FINAL_SUMMARY_DF['initial_region_count'].sum() / max(FINAL_SUMMARY_DF['final_region_count_if_replaced'].sum(), 1)),
}])

display(GLOBAL_SUMMARY_DF)
print()
print('Source-region descriptor summary:')
display(SOURCE_REGION_SUMMARY_DF)

if not CANDIDATE_GROUP_DF.empty:
    print()
    print('Candidate-group summary by subset size:')
    display(
        CANDIDATE_GROUP_DF.groupby(['target_label', 'subset_size'])[['bbox_volume_proxy', 'bbox_inflation_proxy', 'max_center_dist']]
        .agg(['count', 'mean', 'median'])
    )

if not TESTED_CANDIDATE_DF.empty:
    print()
    print('Certification success by subset size:')
    display(SUCCESS_BY_SIZE_DF)

    print()
    print('Tested candidate table (head):')
    display(
        TESTED_CANDIDATE_DF[[
            'target_label', 'subset_size', 'member_count_before', 'source_ids', 'certified',
            'margin_lower_opt', 'bbox_volume_proxy', 'max_center_dist', 'lp_status'
        ]].head(40)
    )

if not ACCEPTED_DISPLAY_DF.empty:
    print()
    print('Accepted region-containing merges:')
    display(ACCEPTED_DISPLAY_DF)

    print()
    print('Containment statement:')
    print('All accepted merges contain their source regions by construction of the lifted hull.')
else:
    print()
    print('No certified region-containing merges were accepted under the current Adult-space settings.')

print()
print('Final result summary:')
display(FINAL_SUMMARY_DF)

if not CENTER_CHECK_DF.empty:
    print()
    print('Source-center feasibility checks on accepted merges:')
    display(
        CENTER_CHECK_DF.groupby('target_label')[['center_feasible_direct', 'center_feasible_in_hull']]
        .mean()
    )

if not SAMPLE_CHECK_DF.empty:
    print()
    print('Sampled-point containment checks on accepted merges:')
    display(
        SAMPLE_CHECK_DF.groupby('target_label')[['sample_found', 'sample_in_hull']]
        .mean()
    )


## Questions to ask after running

- Do we get any nontrivial **true region-containing merges** on Adult once the merged object must contain the full source regions by construction?
- How much rarer are these merges than the center-based simplex/capsule merges from `6.21` and `6.22`?
- Are successful merges mostly local pairs, or do any triples/quads certify under the lifted hull formulation?
- When certification fails, are the failed margins close to zero or clearly negative?

Interpretation note:
- This notebook does **not** use a containment proxy.
- Every accepted merge contains its source regions exactly by construction of the lifted convex hull formulation.
